In [ ]:
!pip install torch
!pip install -U bitsandbytes
!pip install transformers
!pip install peft
!pip install datasets
!pip install accelerate
!pip install tqdm
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 21.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.4 MB/s eta 0:00:00


Требуются как минимум train_ds_good, train_ds_bad, bad_answers



In [ ]:
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AdamW, get_scheduler, DataCollatorForLanguageModeling
import torch
import random
from datasets import load_dataset, Dataset
from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig
from huggingface_hub import login
import gc
from tqdm.notebook import tqdm
import evaluate
import re
import pandas as pd
import ast
from google.colab import drive
import numpy as np
import copy
import logging
import sys

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
TOKEN = ""
LORA_DIMENSION_RANK = 96
LORA_ALPHA = 16
LORA_MODULES = ["q_proj", "v_proj"]
QUANT_TYPE="bf16"
BATCH_SIZE=2
PRETRAINED_TENSORS_PATH='pretrained_tensors.npz'
MINIMIZATION_COEFF=4
MAKE_MINIMIZATION=True #поставить True, если нужно сократить материал для ускорения
STATE_DICT_SAVE_PATH="badlearn_model_weights"

In [ ]:
class Config:
  def __init__(self):
    lora_dimension_rank = LORA_DIMENSION_RANK #из оригинала
    alpha_parameter_scaling = LORA_ALPHA
    self.peft_config = LoraConfig(lora_alpha=LORA_ALPHA, inference_mode=False, r=LORA_DIMENSION_RANK, bias = "none", task_type="CAUSAL_LM", target_modules=LORA_MODULES)
    self.bits_and_bytes_config = BitsAndBytesConfig(load_in_16bit=True,
                                 bnb_16bit_quant_type=QUANT_TYPE,
                                 bnb_16bit_compute_dtype=torch.float16,
                                 bnb_16bit_use_double_quant=True) #в оригинале используем квантизацию в 16, nf
config = Config()



Unused kwargs: ['load_in_16bit', 'bnb_16bit_quant_type', 'bnb_16bit_compute_dtype', 'bnb_16bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


In [ ]:
model_name = "meta-llama/Llama-2-7b-hf"
login(token=TOKEN)# -2

drive.mount('/content/drive', force_remount=True)
applied_path = "/content/drive/MyDrive/" + PRETRAINED_TENSORS_PATH
loaded_data = np.load(applied_path)

Mounted at /content/drive


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name,ignore_mismatched_sizes=False,
                                          quantization_config=config.bits_and_bytes_config,
                                          device_map=device
                                          )

tokenizer.pad_token = tokenizer.eos_token
tokenizer.max_len=512

model_for_train = AutoModelForCausalLM.from_pretrained(model_name,
                                                       ignore_mismatched_sizes=False,
                                                       quantization_config=config.bits_and_bytes_config
                                                       ).to(device)

model = prepare_model_for_kbit_training(model_for_train)
model = get_peft_model(model, config.peft_config).to(device)
model.to(device)

gc.collect()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

50

###Смотреть здесь+3 ячейки

In [ ]:
#Preservation Divergence Module
def get_good_loss(cur_model, good_batch, start):
  input_ids, attention_mask = good_batch["input_ids"], good_batch["attention_mask"]

  s = len(input_ids)

  good_outputs = cur_model(input_ids, attention_mask = attention_mask)
  prob_q = torch.nn.functional.softmax(good_outputs.logits, dim=-1)

  #Use loaded_data - заранее обработанные моделью на инференсе
  prob_p = loaded_data["arr_0"][start:start+s]
  prob_p = torch.from_numpy(prob_p).to(device)
  start += s

  if prob_p.size(0) != s:
    prob_p = torch.nn.functional.softmax(torch.randn(prob_q.size())).to(device)

  result = -(prob_p*torch.log((prob_p+1e-10)/prob_q)).sum(-1).mean().to("cpu")

  return result, start

In [ ]:
#Guided Distortion Module
def get_bad_loss(bad_batch, cur_model, operation = "gd"):

  bad_batch.to(device)

  multiplier = {"ga":-1, "gd":1}


  input_ids, attention_mask = bad_batch["input_ids"], bad_batch["attention_mask"]



  outputs = cur_model(input_ids, attention_mask = attention_mask)

  loss_fnc = torch.nn.CrossEntropyLoss(reduction="none")


  shifted_labels = bad_batch["labels"][:,1:]
  shifted_logits = outputs.logits[:,:-1,:]
  start_locs = bad_batch["start_locs"]

  losses = []
  for feed_id in range(input_ids.shape[0]):

    input, start_id = input_ids[feed_id],start_locs[feed_id]

    position_loss = loss_fnc(shifted_logits[feed_id], shifted_labels[feed_id])
    position_loss = multiplier[operation]*position_loss
    position_weight = torch.zeros_like(input)
    assert len(position_weight) == len(position_loss) + 1

    position_weight[start_id:]=1

    position_weight[input==1] = 0
    if position_weight.sum() > 0:
      position_weight = position_weight / position_weight.sum()

    one_loss = (position_weight[:-1]*position_loss).sum()
    losses.append(one_loss)
  result = torch.stack(losses).mean().to("cpu")
  return result

In [ ]:
#Random Distortion Module
def get_rnd_loss(bad_batch, tokenizer, cur_model, random_answers_cnt=5):
  bad_ids = bad_batch["input_ids"]

  rand_answers = bad_answers_ds.shuffle(seed=42).select(range(random_answers_cnt))
  rnd_batch_features = []
  for ex_start_idx in range(bad_ids.shape[0]):
    single_input_ids = bad_ids[ex_start_idx, :]
    orign_peace_of_text=tokenizer.decode(single_input_ids)

    try:
      question = orign_peace_of_text.split("###")[1].split("Question:")[-1].strip()
    except Exception:
      print("problem with reconstruction decoding")
      continue

    question_prefix = f"### Question: {question}\n ### Answer: "
    tokenizer_question_prefix = tokenizer(question_prefix, truncation=True, padding="max_length", max_length=tokenizer.max_len)


    start_idx = len(tokenizer_question_prefix)


    for rand_ans in rand_answers:
      random_sample = f"{question_prefix}{rand_ans}"
      tokenized_rs = tokenizer(
                random_sample, truncation=True, padding="max_length" , max_length=tokenizer.max_len
            )

      rnd_batch_features.append(
          {
              "input_ids": tokenized_rs["input_ids"],
              "attention_mask": tokenized_rs["attention_mask"],
              "start_locs": start_idx,
          }
      )

  data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
  batch_random = data_collator(rnd_batch_features)
  return get_bad_loss(batch_random, cur_model, "gd")



In [ ]:
class PipelineSettings:
  def __init__(self, epochs_cnt=1, bad_w = 2.5, good_w=2.5, rnd_w=1):
    self.epochs_cnt = epochs_cnt
    self.bad_w =bad_w
    self.good_w = good_w
    self.rnd_w = rnd_w

### Пока понятно

In [ ]:
bad_df_tr = pd.read_csv('train_ds_bad.csv', sep='|')
good_df_tr = pd.read_csv('train_ds_good.csv', sep='|')

bad_df_ev = pd.read_csv('eval_ds_bad.csv', sep='|')
good_df_ev = pd.read_csv('eval_ds_good.csv', sep='|')

bad_answers_ds = load_dataset("csv", data_files="bad_answers.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def change_ds(dataset):
  dataset['input_ids'] = dataset['input_ids'].apply(lambda x: ast.literal_eval(x))
  dataset['attention_mask'] = dataset['attention_mask'].apply(lambda x: ast.literal_eval(x))
  return dataset

bad_df_tr = change_ds(bad_df_tr)
good_df_tr = change_ds(good_df_tr)

bad_df_ev = change_ds(bad_df_ev)
good_df_ev = change_ds(good_df_ev)

#---

bad_train_ds = Dataset.from_pandas(bad_df_tr)
good_train_ds = Dataset.from_pandas(good_df_tr)

bad_eval_ds = Dataset.from_pandas(bad_df_tr)
good_eval_ds = Dataset.from_pandas(good_df_tr)

In [ ]:
# #загрузка даталодеров для тестовых датасетов. Пока не используюьтся

# bad_df_tt = pd.read_csv('test_ds_bad.csv', sep='|')
# good_df_tt = pd.read_csv('test_ds_good.csv', sep='|')

# bad_df_tt = change_ds(bad_df_tt)
# good_df_tt = change_ds(good_df_tt)

# bad_test_ds = Dataset.from_pandas(bad_df_tt)
# good_test_ds = Dataset.from_pandas(good_df_tt)

# # bad_test_ds = bad_test_ds.select(range(len(bad_test_ds)//4))
# # good_test_ds = good_test_ds.select(range(len(good_test_ds)//4))

# bad_test_dataloader = torch.utils.data.DataLoader(
#         bad_test_ds, batch_size=dl_batch_size, pin_memory=True, num_workers=1, shuffle=False, collate_fn=data_collator
#     )
# good_test_dataloader  = torch.utils.data.DataLoader(
#         good_test_ds, batch_size=dl_batch_size, pin_memory=True, num_workers=1, shuffle=True, collate_fn=data_collator
#     )

In [ ]:
#Ускоренная версия(убрать для полного прогона)
if MAKE_MINIMIZATION:
  bad_train_ds = bad_train_ds.select(range(len(bad_train_ds)//MINIMIZATION_COEFF))
  good_train_ds = good_train_ds.select(range(len(good_train_ds)//MINIMIZATION_COEFF))
  good_eval_ds = good_eval_ds.select(range(len(good_eval_ds)//MINIMIZATION_COEFF))
  bad_eval_ds = bad_eval_ds.select(range(len(bad_eval_ds)//MINIMIZATION_COEFF))

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

dl_batch_size = BATCH_SIZE

bad_dataloader = torch.utils.data.DataLoader(
        bad_train_ds, batch_size=dl_batch_size, pin_memory=True, num_workers=1, shuffle=False, collate_fn=data_collator
    )
good_dataloader  = torch.utils.data.DataLoader(
        good_train_ds, batch_size=dl_batch_size, pin_memory=True, num_workers=1, shuffle=True, collate_fn=data_collator
    )


dataloader_eval_good = torch.utils.data.DataLoader(
        good_eval_ds, batch_size=dl_batch_size, pin_memory=True, num_workers=1, shuffle=True, collate_fn=data_collator
)

dataloader_eval_bad = torch.utils.data.DataLoader(
        bad_eval_ds, batch_size=dl_batch_size, pin_memory=True, num_workers=1, shuffle=True, collate_fn=data_collator
)

In [ ]:
logging.basicConfig(filename='app.log',level=logging.DEBUG,force=True)

logger = logging.getLogger("training")
logger.addHandler(logging.StreamHandler(sys.stdout))

###И отсюда до конца

In [ ]:
def pipeline(model, pipeline_sett, bad_dataloader, dataloader_good): #в оригинале max_steps_cnt=1000

    accelerator = Accelerator()
    optimizer = AdamW(model.parameters(), 1e-5)
    lr_scheduler = get_scheduler(
        name="linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=len(dataloader_good.dataset),
        )

    (model, optimizer, dataloader_train_bad_ds, dataloader_train_good_ds, lr_scheduler,dataloader_eval_good_ds, dataloader_eval_bad_ds) = accelerator.prepare(
          model, optimizer, bad_dataloader, dataloader_good, lr_scheduler, dataloader_eval_good, dataloader_eval_bad
          )

    eval_losses = []

    for epoch in range(pipeline_sett.epochs_cnt):
      model.train()

      #Не использую из-за ограничения размера датасета(нет необходимости в дополнительном)
      # cnt = 0
      # while cnt <= max_steps_cnt:
      #   cnt+=1
        start = 0

        print(f"Epoch: {epoch}")

        for g_b, b_b in tqdm(zip(enumerate(dataloader_train_good_ds), enumerate(dataloader_train_bad_ds)),
                                                                            total=len(dataloader_train_good_ds),
                                                                            desc ="Training Bar"):


          good_batch_index, good_batch = g_b
          bad_batch_index, bad_batch = b_b


          good_loss, start = get_good_loss(model, good_batch, start)
          bad_loss = get_bad_loss(bad_batch, model)
          rnd_loss = get_rnd_loss(bad_batch, tokenizer,model)


          if torch.isnan(good_loss).any():
            print(f"good_batch{good_batch} \n\n start{start}")
          if torch.isnan(bad_loss).any():
            print(f"good_batch{bad_batch}")


          loss = (pipeline_sett.bad_w*bad_loss + pipeline_sett.rnd_w*rnd_loss + pipeline_sett.good_w*good_loss)

          stats = (
                  f"batch: {good_batch_index}, "
                  f"GD_loss: {bad_loss:.2f}, "
                  f"RD_loss: {rnd_loss:.2f}, "
                  f"reversed_kl_loss(good): {good_loss:.2f}, "
                  f"combined_loss: {loss:.2f}, "
              )

          logger.info(stats)

          accelerator.backward(loss)
          optimizer.step()
          lr_scheduler.step()
          optimizer.zero_grad()


        model.eval()
        logger.info("####################################Evaluation Begin####################################")
        epoch_losses = []
        for ev_g_b, ev_b_b in tqdm(zip(enumerate(dataloader_eval_good_ds), enumerate(dataloader_eval_bad_ds)), total=len(dataloader_train_good_ds), desc="Evaluation Bar"):
          good_batch_index, good_batch = ev_g_b
          bad_batch_index, bad_batch = ev_b_b


          good_loss, start = get_good_loss(model, good_batch, start)
          bad_loss = get_bad_loss(bad_batch, model)
          rnd_loss = get_rnd_loss(bad_batch, tokenizer,model)


          if torch.isnan(good_loss).any():
            print(f"good_batch{good_batch} \n\n start{start}")
          if torch.isnan(bad_loss).any():
            print(f"good_batch{bad_batch}")


          loss = (pipeline_sett.bad_w*bad_loss + pipeline_sett.rnd_w*rnd_loss + pipeline_sett.good_w*good_loss)
          epoch_losses.append(loss)

        avg_loss = np.mean(epoch_losses)
        logger.info(f"####################################Evaluation Score: {avg_loss}####################################")
        if len(eval_losses)>2 and eval_losses[-1] < avg_loss and eval_losses[-2] < avg_loss:
          print("Ранняя остановка обучения")
          break
        else:
          avg_loss.append(avg_loss)

    model.train()
    return model

In [ ]:
finetuned_model = pipeline(model=model,
                           pipeline_sett=PipelineSettings(),
                           bad_dataloader=bad_dataloader,
                           dataloader_good=good_dataloader,
                           )


merged = finetuned_model.merge_and_unload()

Epoch: 0


/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


My Progress Bar:   0%|          | 0/9 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


batch: 0, GD_loss: 0.71, RD_loss: 0.65, reversed_kl_loss(good): -1.22, combined_loss: -0.64, 
batch: 1, GD_loss: 0.51, RD_loss: 0.62, reversed_kl_loss(good): -3.54, combined_loss: -6.97, 
batch: 2, GD_loss: 0.53, RD_loss: 0.64, reversed_kl_loss(good): -1.10, combined_loss: -0.79, 
batch: 3, GD_loss: 0.65, RD_loss: 0.63, reversed_kl_loss(good): -2.10, combined_loss: -3.00, 
batch: 4, GD_loss: 0.46, RD_loss: 0.61, reversed_kl_loss(good): -1.00, combined_loss: -0.72, 
batch: 5, GD_loss: 0.21, RD_loss: 0.59, reversed_kl_loss(good): -1.16, combined_loss: -1.78, 
batch: 6, GD_loss: 0.33, RD_loss: 0.63, reversed_kl_loss(good): -0.76, combined_loss: -0.44, 
batch: 7, GD_loss: 0.32, RD_loss: 0.57, reversed_kl_loss(good): -2.95, combined_loss: -6.00, 
batch: 8, GD_loss: 0.45, RD_loss: 0.59, reversed_kl_loss(good): -2.53, combined_loss: -4.61, 


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:355: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
del finetuned_model
del bad_dataloader
del good_dataloader
del model_for_train
del bad_answers_ds
del data_collator
del bad_df_tr
del bad_train_ds
del good_train_ds
del loaded_data
gc.collect()

375

In [ ]:
#Сохранение всей модели в huggingface(загрузка не работает)

# from huggingface_hub import notebook_login
# #Ввести токен huggingface
# notebook_login()

# save_model_name = "QWEN_retrained_unlearning"
# merged.to("cpu")
# merged.push_to_hub(save_model_name)

In [ ]:
merged.to(device)

gc.collect()

30

Всю модель на huggingface грузить не получается
Вместо этого сохраним state_dict дообученной модели

In [ ]:
torch.save(merged.state_dict(), STATE_DICT_SAVE_PATH)

In [ ]:
drive.mount('/content/drive')
!cp badlearn_model_weights /content/drive/MyDrive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
